# K1 · Series inventory (CPU)
Settings: **Internet ON**, accelerator **None**, competition data attached.

Outputs (in `/kaggle/working`, saved with *Save Version* and reused as input by K3, K4 and K5):
* `knee_common.py`: shared code;
* `wheels/`: JPEG decoders for offline inference;
* `series_inventory.csv`: one row per series;
* `slices.parquet`: sorted slice order of every series;
* `check_grid.png`: visual check.

In [ ]:
# Decoders for compressed DICOM (JPEG Lossless, JPEG 2000) + offline copy for K5
!pip -q install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
!pip -q download --no-deps -d /kaggle/working/wheels pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
!ls /kaggle/working/wheels

In [ ]:
%%writefile /kaggle/working/knee_common.py
# knee_common.py
# Shared code for K1, K3, K4 and K5 (RSNA Knee Abnormality Detection).
# K1 writes this file to /kaggle/working, so it becomes part of K1's output,
# which the following kernels attach as input and import.

import os
import glob
import numpy as np
import pandas as pd

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

# ---------------------------------------------------------------- paths

def find_comp_dir():
    """Locate the mounted competition folder, whatever the Kaggle path is."""
    candidates = [
        "/kaggle/input/competitions/rsna-knee-abnormality-detection",
        "/kaggle/input/rsna-knee-abnormality-detection",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")) or os.path.exists(os.path.join(c, "test.csv")):
            return c
    for pattern in ("/kaggle/input/*/test.csv", "/kaggle/input/*/*/test.csv"):
        hits = glob.glob(pattern)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("Competition data not found: add it as an input of the notebook.")


def find_input_file(name):
    """Find a file produced by another kernel (attached as input)."""
    for pattern in (f"/kaggle/input/*/{name}", f"/kaggle/input/*/*/{name}",
                    f"/kaggle/input/*/*/*/{name}", f"/kaggle/working/{name}"):
        hits = glob.glob(pattern)
        if hits:
            return hits[0]
    return None


def series_csv_path(comp, split):
    return os.path.join(comp, f"{split}_series.csv")


def series_path(comp, split, study, series):
    return os.path.join(comp, f"{split}_series", str(study), str(series))

# ---------------------------------------------------------------- DICOM reading

HEADER_TAGS = ["InstanceNumber", "ImagePositionPatient", "ImageOrientationPatient",
               "PixelSpacing", "Rows", "Columns"]


def _to_floats(v, n):
    try:
        out = [float(x) for x in v]
        return out if len(out) == n else None
    except Exception:
        return None


def read_series_headers(path):
    """Read the headers of every slice of a series and sort them in space.

    Returns a sorted DataFrame (file, sort key, instance number, pixel spacing,
    rows, columns, transfer syntax)."""
    import pydicom
    rows = []
    files = sorted(f for f in os.listdir(path) if f.endswith(".dcm"))
    for f in files:
        try:
            ds = pydicom.dcmread(os.path.join(path, f), stop_before_pixels=True,
                                 specific_tags=HEADER_TAGS)
        except Exception:
            continue
        ipp = _to_floats(getattr(ds, "ImagePositionPatient", None) or [], 3)
        iop = _to_floats(getattr(ds, "ImageOrientationPatient", None) or [], 6)
        ps = _to_floats(getattr(ds, "PixelSpacing", None) or [], 2)
        inst = getattr(ds, "InstanceNumber", None)
        try:
            inst = int(inst) if inst is not None else None
        except Exception:
            inst = None
        key = np.nan
        if ipp is not None and iop is not None:
            normal = np.cross(np.array(iop[:3]), np.array(iop[3:]))
            key = float(np.dot(np.array(ipp), normal))
        ts = ""
        try:
            ts = str(ds.file_meta.TransferSyntaxUID)
        except Exception:
            pass
        rows.append(dict(
            file=f, key=key, inst=inst,
            ps_r=ps[0] if ps else np.nan, ps_c=ps[1] if ps else np.nan,
            rows=int(getattr(ds, "Rows", 0) or 0), cols=int(getattr(ds, "Columns", 0) or 0),
            ts=ts,
        ))
    df = pd.DataFrame(rows, columns=["file", "key", "inst", "ps_r", "ps_c", "rows", "cols", "ts"])
    if len(df) == 0:
        return df
    if df["key"].notna().all() and df["key"].nunique() > 1:
        df = df.sort_values("key")
    elif df["inst"].notna().all():
        df = df.sort_values("inst")
    return df.reset_index(drop=True)


def read_pixels(fp):
    """Read one slice as a 2D float32 array."""
    import pydicom
    ds = pydicom.dcmread(fp)
    arr = ds.pixel_array.astype(np.float32)
    if arr.ndim == 3:          # rare: colour or multi-frame, keep the first plane
        arr = arr[..., 0] if arr.shape[-1] in (3, 4) else arr[0]
    slope = float(getattr(ds, "RescaleSlope", 1) or 1)
    inter = float(getattr(ds, "RescaleIntercept", 0) or 0)
    return arr * slope + inter

# ---------------------------------------------------------------- canonical volume

# (name, plane, filter, number of slices); 64 slices in total, always in this order
SLOTS = [
    ("sag_fluid", "Sagittal", "fluid", 18),
    ("sag_anat", "Sagittal", "anat", 14),
    ("cor_fluid", "Coronal", "fluid", 12),
    ("cor_anat", "Coronal", "anat", 8),
    ("axial", "Axial", "any", 12),
]
N_SLICES = sum(s[3] for s in SLOTS)          # 64
CROP_MM = 140.0
LO, HI = 0.06, 0.94


def choose_series(study_series):
    """Pick one series per slot from the provided series descriptors.

    study_series: rows of the series CSV for one study, plus an optional
    column n (number of slices) when known.
    Returns a list of 5 series UIDs (None when the slot is empty).
    A series from another plane is never used to fill a gap."""
    df = study_series.copy()
    if "n" not in df.columns:
        df["n"] = 0
    df = df[df["n"].fillna(0) >= 3] if (df["n"] > 0).any() else df
    used, out = set(), []
    for _, plane, kind, _k in SLOTS:
        cand = df[(df["Anatomical_Plane"] == plane) & (~df["SeriesInstanceUID"].isin(used))]
        if kind == "fluid":
            cand = cand[cand["Fluid_Sensitive"] == 1]
            cand = cand.assign(score=cand["Fat_Suppression"] * 1000 + cand["n"].clip(upper=200))
        elif kind == "anat":
            cand = cand[cand["Fat_Suppression"] == 0]
            cand = cand.assign(score=(1 - cand["Fluid_Sensitive"]) * 1000 + cand["n"].clip(upper=200))
        else:
            cand = cand.assign(score=cand["Fluid_Sensitive"] * cand["Fat_Suppression"] * 1000
                               + cand["n"].clip(upper=200))
        if len(cand) == 0:
            out.append(None)
            continue
        best = cand.sort_values("score", ascending=False).iloc[0]["SeriesInstanceUID"]
        used.add(best)
        out.append(best)
    return out


def pick_indices(n, k, lo=LO, hi=HI):
    """k evenly spaced indices between lo and hi of the stack (edges included)."""
    pos = np.linspace(lo, hi, k)
    return np.clip(np.round(pos * (n - 1)).astype(int), 0, n - 1)


def crop_mm(img, ps_r, ps_c, size, crop_mm=CROP_MM):
    """Square of crop_mm millimetres around the centre, zero-padded when the
    field of view is smaller, then resized to size x size."""
    import cv2
    if not (ps_r and ps_c) or np.isnan(ps_r) or np.isnan(ps_c) or ps_r <= 0 or ps_c <= 0:
        ps_r = ps_c = crop_mm / max(img.shape)   # fallback: keep the whole image
    hr = int(round(crop_mm / 2.0 / ps_r))
    hc = int(round(crop_mm / 2.0 / ps_c))
    H, W = img.shape
    cy, cx = H // 2, W // 2
    canvas = np.zeros((2 * hr, 2 * hc), dtype=np.float32)
    y0, y1 = max(0, cy - hr), min(H, cy + hr)
    x0, x1 = max(0, cx - hc), min(W, cx + hc)
    oy, ox = y0 - (cy - hr), x0 - (cx - hc)
    canvas[oy:oy + (y1 - y0), ox:ox + (x1 - x0)] = img[y0:y1, x0:x1]
    interp = cv2.INTER_AREA if canvas.shape[0] > size else cv2.INTER_LINEAR
    return cv2.resize(canvas, (size, size), interpolation=interp)


def normalize_to_uint8(stack):
    """Per-series normalisation: percentiles 1 and 99, then 0 to 255."""
    lo, hi = np.percentile(stack, [1, 99])
    stack = np.clip((stack - lo) / max(hi - lo, 1e-6), 0, 1)
    return (stack * 255).astype(np.uint8)


def build_volume(comp, split, study, study_series, size=224, sorted_files=None):
    """Build the canonical volume of one study.

    sorted_files: optional dict {series: (sorted file list, ps_r, ps_c)} from the
    K1 inventory; otherwise headers are read again.
    Returns (uint8 volume of shape (64, size, size), bool mask (64,), chosen series)."""
    vol = np.zeros((N_SLICES, size, size), dtype=np.uint8)
    mask = np.zeros(N_SLICES, dtype=bool)
    chosen = choose_series(study_series)
    start = 0
    for (name, plane, kind, k), sid in zip(SLOTS, chosen):
        if sid is not None:
            try:
                p = series_path(comp, split, study, sid)
                if sorted_files is not None and sid in sorted_files:
                    files, ps_r, ps_c = sorted_files[sid]
                else:
                    h = read_series_headers(p)
                    files = h["file"].tolist()
                    ps_r = float(h["ps_r"].median()) if len(h) else np.nan
                    ps_c = float(h["ps_c"].median()) if len(h) else np.nan
                if len(files) > 0:
                    idx = pick_indices(len(files), k)
                    cache, sl = {}, []
                    for i in idx:
                        if i not in cache:
                            cache[i] = crop_mm(read_pixels(os.path.join(p, files[i])), ps_r, ps_c, size)
                        sl.append(cache[i])
                    vol[start:start + k] = normalize_to_uint8(np.stack(sl))
                    mask[start:start + k] = True
            except Exception as e:
                print(f"[volume] {study} {sid} : {type(e).__name__} {e}")
        start += k
    return vol, mask, chosen

# ---------------------------------------------------------------- 2.5D windows

def window_centers(mask):
    """Centres c such that c-1, c, c+1 are valid and inside the same slot."""
    bounds, s = [], 0
    for _, _, _, k in SLOTS:
        bounds.append((s, s + k))
        s += k
    centers = []
    for a, b in bounds:
        for c in range(a + 1, b - 1):
            if mask[c - 1] and mask[c] and mask[c + 1]:
                centers.append(c)
    return np.array(centers, dtype=int)


def make_windows(vol, centers):
    """(K, 3, H, W) from the centres."""
    return np.stack([vol[c - 1:c + 2] for c in centers]) if len(centers) else np.zeros((0, 3) + vol.shape[1:], vol.dtype)

In [ ]:
import os, sys, time, numpy as np, pandas as pd
from multiprocessing import Pool
sys.path.insert(0, "/kaggle/working")
import importlib, knee_common as kc
importlib.reload(kc)
import pydicom
print("pydicom", pydicom.__version__)

COMP = kc.find_comp_dir()
print("COMP =", COMP)
train = pd.read_csv(f"{COMP}/train.csv")
ser = pd.read_csv(kc.series_csv_path(COMP, "train"))
print(train.shape, ser.shape)
print(ser.head())

In [ ]:
def inventory_one(args):
    study, series = args
    p = kc.series_path(COMP, "train", study, series)
    try:
        h = kc.read_series_headers(p)
    except Exception as e:
        return series, None, f"{type(e).__name__}: {e}"
    return series, h, None

jobs = list(zip(ser["StudyInstanceUID"], ser["SeriesInstanceUID"]))
t0 = time.time()
inv_rows, slice_rows, errors = [], [], []
with Pool(os.cpu_count()) as pool:
    for i, (sid, h, err) in enumerate(pool.imap_unordered(inventory_one, jobs, chunksize=8)):
        if err or h is None or len(h) == 0:
            errors.append((sid, err))
            inv_rows.append(dict(SeriesInstanceUID=sid, n=0))
        else:
            inv_rows.append(dict(
                SeriesInstanceUID=sid, n=len(h),
                ps_r=float(h["ps_r"].median()), ps_c=float(h["ps_c"].median()),
                rows=int(h["rows"].median()), cols=int(h["cols"].median()),
                ts=h["ts"].mode().iloc[0], sorted_by_position=bool(h["key"].notna().all()),
            ))
            slice_rows.append(pd.DataFrame({"SeriesInstanceUID": sid, "rank": np.arange(len(h)), "file": h["file"].values}))
        if (i + 1) % 2000 == 0:
            print(f"{i+1}/{len(jobs)} series  {time.time()-t0:.0f} s")
print(f"done in {time.time()-t0:.0f} s, errors: {len(errors)}")

inv = ser.merge(pd.DataFrame(inv_rows), on="SeriesInstanceUID", how="left")
inv.to_csv("/kaggle/working/series_inventory.csv", index=False)
pd.concat(slice_rows, ignore_index=True).to_parquet("/kaggle/working/slices.parquet", index=False)
print(errors[:10])

In [ ]:
# Sanity statistics
print("Series per study:")
print(inv.groupby("StudyInstanceUID").size().describe())
print("\nPlanes:"); print(inv["Anatomical_Plane"].value_counts())
print("\nFluid sensitive x fat suppression:")
print(pd.crosstab(inv["Fluid_Sensitive"], inv["Fat_Suppression"]))
print("\nNumber of slices:"); print(inv["n"].describe())
print("\nPixel spacing (mm):"); print(inv["ps_r"].describe())
print("\nTransfer syntaxes:"); print(inv["ts"].value_counts())
print("\nSorted by spatial position:", inv["sorted_by_position"].mean())

In [ ]:
# Coverage of the 5 slots of the canonical volume
slots = {s[0]: 0 for s in kc.SLOTS}
for study, g in inv.groupby("StudyInstanceUID"):
    for (name, *_), sid in zip(kc.SLOTS, kc.choose_series(g)):
        slots[name] += sid is not None
n_st = inv["StudyInstanceUID"].nunique()
for k, v in slots.items():
    print(f"{k:10s} filled in {v}/{n_st} studies ({100*v/n_st:.1f} %)")

In [ ]:
# Decoding test: a few series per transfer syntax
for ts, g in inv[inv["n"] > 0].groupby("ts"):
    ok, fail = 0, []
    for _, r in g.sample(min(5, len(g)), random_state=0).iterrows():
        p = kc.series_path(COMP, "train", r.StudyInstanceUID, r.SeriesInstanceUID)
        f = sorted(os.listdir(p))[0]
        try:
            a = kc.read_pixels(os.path.join(p, f)); ok += 1
        except Exception as e:
            fail.append(f"{type(e).__name__}: {e}")
    print(ts, "ok", ok, "failures", len(fail), fail[:1])

In [ ]:
# Visual check: 6 studies, middle slice of each slot
import matplotlib.pyplot as plt
studies = inv["StudyInstanceUID"].drop_duplicates().sample(6, random_state=1).tolist()
fig, axes = plt.subplots(6, 5, figsize=(15, 18))
for i, st in enumerate(studies):
    vol, mask, chosen = kc.build_volume(COMP, "train", st, inv[inv.StudyInstanceUID == st], size=224)
    s = 0
    for j, (name, plane, kind, k) in enumerate(kc.SLOTS):
        axes[i, j].imshow(vol[s + k // 2], cmap="gray"); axes[i, j].axis("off")
        axes[i, j].set_title(f"{name} {'ok' if mask[s] else 'empty'}", fontsize=9)
        s += k
plt.tight_layout(); plt.savefig("/kaggle/working/check_grid.png", dpi=70); plt.show()